In [1]:
import spacy
import pandas as pd 
import jsonlines
import spacy
from tqdm.auto import tqdm
from more_itertools import flatten
from unidecode import unidecode
import re
import dateparser

In [2]:
def get_ents(text_list):
    ents = []
    for doc in tqdm(
        spacy_model.pipe(
            text_list,
            disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"]
        ), total=len(text_list)
    ):
        doc_ents = list(doc.ents)
        ents.append(doc_ents)
    return ents


spacy_model = spacy.load('en_core_web_lg')
agenda_watch_df = pd.read_csv('../data/agenda-watch-bay-area-202027.csv', index_col=0)

In [ ]:
with jsonlines.open('../data/sfchron-fetched-articles.jsonl') as f:
    sfchron_data = list(f)

sfchron_article_df = pd.DataFrame(sfchron_data)

In [3]:
# import pickle
# with open('cache/sfchron_ents.pkl', 'rb') as f:
#     list_of_sfcrhon_ents = pickle.load(f)

In [ ]:
sfchron_ents = get_ents(sfchron_article_df['article_text'])
list_of_sfcrhon_ents = list(map(lambda x: list(map(lambda y: (str(y), y.label_), x)), sfchron_ents))

In [6]:
list_of_sfcrhon_ents[0]

[('Kris Perry', 'PERSON'),
 ('Sandy Stier', 'PERSON'),
 ('San Francisco', 'GPE'),
 ('Gibson Dunn', 'PERSON'),
 ('Thursday, March 21, 2013', 'DATE'),
 ('Washington', 'GPE'),
 ('the Supreme Court', 'ORG'),
 ('this week', 'DATE'),
 ('two', 'CARDINAL')]

In [8]:
agenda_watch_ents = get_ents(agenda_watch_df['text'])
list_of_agenda_watch_ents = list(map(lambda x: list(map(lambda y: (str(y), y.label_), x)), agenda_watch_ents))

  0%|          | 0/1073 [00:00<?, ?it/s]

In [9]:
all_agenda_watch_ents = list(flatten(list_of_agenda_watch_ents))

In [11]:
agenda_watch_dates = list(filter(lambda x: x[1] == 'DATE', all_agenda_watch_ents))
agenda_watch_parsed_dates = list(map(lambda x: dateparser.parse(x[0]), agenda_watch_dates))

agenda_watch_times = list(filter(lambda x: x[1] == 'TIME', all_agenda_watch_ents))
agenda_watch_parsed_times = list(map(lambda x: dateparser.parse(x[0]), agenda_watch_times))

In [26]:
stopwords = ['san francisco', 'california', 'ca', 'oakland',]
ent_types_to_exclude = ['CARDINAL', 'PRODUCT', 'TIME', 'DATE', 'WORK_OF_ART']

In [27]:
all_agenda_watch_ents_to_match = list(filter(
    lambda x: x[1] not in ent_types_to_exclude, all_agenda_watch_ents
))

In [28]:
import re 
from jellyfish import jaro_similarity
from polyleven import levenshtein

def get_num_matches(ents_a, ents_b):
    match_pairs = []
    for e_a in ents_a:
        for e_b in ents_b:
            if levenshtein(e_a, e_b, 2) < 3:
                match_pairs.append((e_a, e_b))
    return match_pairs

In [75]:
sfchron_to_agendawatch = []

def process_ents_in_doc(list_of_ents):
    possible_ents = list(filter(lambda x: x[1] not in ent_types_to_exclude, list_of_ents))
    possible_ents = list(map(lambda x: x[0], possible_ents))
    possible_ents = list(map(lambda x: re.sub('\s+', ' ', x).lower(), possible_ents))
    possible_ents = list(filter(lambda x: x not in stopwords, possible_ents))
    return possible_ents


processed_agendawatch_ents = list(map(process_ents_in_doc, list_of_agenda_watch_ents))
all_processed_agendawatch_ents = list(flatten(processed_agendawatch_ents))

for sfchron_doc_ents in tqdm(list_of_sfcrhon_ents):
    num_matches_with_agenda_watch = []
    processed_sfchron = process_ents_in_doc(sfchron_doc_ents)
    all_sfchron_matches = get_num_matches(processed_sfchron, all_processed_agendawatch_ents)
    all_sfchron_matches = list(set(map(lambda x: x[0], all_sfchron_matches)))
    for processed_agendawatch in processed_agendawatch_ents:
        num_matches = get_num_matches(all_sfchron_matches, processed_agendawatch)
        num_matches_with_agenda_watch.append(num_matches)
    sfchron_to_agendawatch.append(num_matches_with_agenda_watch)

  0%|          | 0/140493 [00:00<?, ?it/s]

In [80]:
import json

In [90]:
all_dicts = []
for one_sfchron in tqdm(sfchron_to_agendawatch):
    one_line_dict = {}
    for i, l in enumerate(one_sfchron):
        if len(l) > 0:
            one_line_dict[i] = l
    all_dicts.append(one_line_dict)

  0%|          | 0/140493 [00:00<?, ?it/s]

In [92]:
json.dump(all_dicts, open('cache/2023-03-15__sfchron-to-agenda-watch-old-minutes.json', 'w'))

In [2]:
import requests

In [3]:
url = 'https://cupertino.legistar.com/View.ashx?M=M&ID=337959&GUID=A0F19CF3-BD77-44F8-BE6E-E76673FAAFDA'
resp = requests.get(url)

In [17]:
import docx2txt

In [18]:
docx2txt.process(BytesIO(c))

KeyError: "There is no item named 'word/document.xml' in the archive"

In [19]:
# text = textract.process(resp.content)

In [12]:
import mimetypes

In [18]:
mimetypes.guess_extension(resp.headers['Content-Type'])

'.doc'

In [34]:
from tqdm.auto import tqdm

In [30]:
import glob
import pandas as pd 

input_files = glob.glob('../bin/aw-legistar-scraper-selenium/legistar_scraper/output-*/*')
all_dfs = []
for csv_file in input_files:
    one_df = pd.read_csv(csv_file, index_col=0)
    all_dfs.append(one_df)
files_to_get = pd.concat(all_dfs)

In [38]:
requests.head(url).headers

{'Cache-Control': 'private', 'Content-Length': '143', 'Content-Type': 'text/html; charset=utf-8', 'Location': '/Confirmation.aspx?M1=Gone', 'Server': 'Microsoft-IIS/10.0', 'p3p': 'CP="IDC DSP COR ADM DEVi TAIi PSA PSD IVAi IVDi CONi HIS OUR IND CNT"', 'X-Powered-By': 'ASP.NET', 'GranicusServer': 'gasmp-insweb2', 'Date': 'Mon, 20 Mar 2023 21:01:50 GMT', 'Set-Cookie': 'BIGipServerinsite.legistar.com_443=924975882.47873.0000; path=/; Httponly; Secure'}

In [ ]:
headers = []
for url in tqdm(files_to_get.sample(len(files_to_get))['url']):
    resp = requests.head(url)
    headers.append(resp.headers)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def get_headers(url):
    try:
        response = requests.head(url)
        return response.headers['Content-Type']
    except:
        print('failed...')
        return None

headers = []
n_files = len(files_to_get)
with ThreadPoolExecutor(max_workers=10) as executor:
    for output in tqdm(executor.map(get_headers, files_to_get.sample(n_files)['url']), total=n_files):
        if output is not None:
            headers.append(output)

In [48]:
pd.Series(headers).value_counts()

application/pdf             9169
text/html; charset=utf-8      20
text/html                      9
application/msword             8
application/octet-stream       1
dtype: int64

In [50]:
mimetypes.guess_extension('application/octet-stream')

'.bin'

In [54]:
(pd.DataFrame(all_agenda_watch_ents_to_match)[0]
 .str.replace('\s+', ' ', regex=True)
 .str.lower()
 .value_counts()
 .index
)

Index(['council', 'city', 'san francisco', 'california', 'ca', 'city hall',
       'oakland', 'board', 'frank h. ogawa plaza', 'oakland city hall',
       ...
       'councilmember handbook review',
       'the cellular service equipment leases', 'amendment no. 1',
       't-mobile west corporation', 'verizon wireless',
       'the cellular service equipment sites leases', '4/15/16', '4/1/16',
       'program administratorâs evidence of financial responsibility - 2016',
       'el camino real plan advisory committee'],
      dtype='object', length=10129)